You can download and run this notebook locally, or you can run it for free in a cloud environment using Colab or Sagemaker Studio Lab:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kirbyju/TCIA_Notebooks/blob/main/CMMD/CMMD_Clinical.ipynb)

[![Open In SageMaker Studio Lab](https://studiolab.sagemaker.aws/studiolab.svg)](https://studiolab.sagemaker.aws/import/github.com/kirbyju/TCIA_Notebooks/blob/main/CMMD/CMMD_Clinical.ipynb)

# Summary

[The Cancer Imaging Archive (TCIA)](https://www.cancerimagingarchive.net/) is a public service funded by the National Cancer Institute that hosts and de-identifies medical images so researchers can share them.

[The Chinese Mammography Database (CMMD)](https://doi.org/10.7937/tcia.eqde-4b16) holds digital mammograms of 1,775 patients, with a clinical spreadsheet that gives each lesion's laterality, the patient's age, the type of abnormality (mass, calcification or both), whether biopsy found it benign or malignant, and, for a subset of malignant cases, the molecular subtype (Luminal A, Luminal B, HER2-enriched or triple negative). Both the images and the spreadsheet are released under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/), so no login is needed.

In this notebook we will:

1. Use the TCIA website API to look up the collection and find its clinical data file.
2. Join the clinical spreadsheet to the image inventory on the patient ID.
3. Download one patient for each molecular subtype (about 70 MB) and join each image to its clinical row by patient and breast side, since the spreadsheet has one row per affected breast rather than one per patient.
4. Show the four mammograms of each patient, labeled with the clinical data that belongs to that breast.

Please be sure to abide by the [TCIA Data Usage Policy and Restrictions](https://www.cancerimagingarchive.net/data-usage-policies-and-restrictions/) by citing the dataset anywhere these data are discussed:

```Cui, Chunyan; Li Li; Cai, Hongmin; Fan, Zhihao; Zhang, Ling; Dan, Tingting; Li, Jiao; Wang, Jinghua. (2021) The Chinese Mammography Database (CMMD): An online mammography database with biopsy confirmed types for machine diagnosis of breast. The Cancer Imaging Archive. DOI: https://doi.org/10.7937/tcia.eqde-4b16```

# Setup
Install [tcia_utils](https://pypi.org/project/tcia-utils/) to make it easier to access TCIA data via its APIs. It also installs `pydicom`, which we use to read the image headers and pixels. `openpyxl` lets pandas read the clinical spreadsheet.

In [ ]:
import sys

!{sys.executable} -m pip install --upgrade -q tcia_utils openpyxl

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import pydicom
from tcia_utils import idc
from tcia_utils import wordpress

# set logging level to INFO in Google Colab (not necessary in Jupyter)
if 'google.colab' in sys.modules:
  import logging

  for handler in logging.root.handlers[:]:
      logging.root.removeHandler(handler)

  # Set handler with level = info
  logging.basicConfig(format='%(asctime)s:%(levelname)s:%(message)s',
                      level=logging.INFO)

  print("Google Colab Logging set to INFO")

# Find the clinical data on the TCIA website

Every TCIA collection page lists its downloads, and the **wordpress** module of tcia_utils reads those pages through the TCIA website API. Here we pull the CMMD entry and list its downloads with their license and file type.

In [ ]:
collections = wordpress.getCollections(format="df")
cmmd = collections[collections["slug"] == "cmmd"].iloc[0]

print(cmmd["collection_title"], "-", cmmd["subjects"], "subjects")
print("DOI:", cmmd["collection_doi"])

downloads = pd.DataFrame([
    {
        "download type": ", ".join(d["download type"]),
        "file type": ", ".join(d["file type"]),
        "license": d["license"]["label"],
        "download url": d["download url"],
    }
    for d in cmmd["collection_downloads"]
])
downloads

The images are DICOM, and the clinical data is a small XLSX file. We can read it straight from its URL.

In [ ]:
clinical_url = downloads.loc[downloads["download type"] == "Clinical Data", "download url"].iloc[0]
clinical = pd.read_excel(clinical_url)

print(len(clinical), "rows for", clinical["ID1"].nunique(), "patients")
clinical.head()

There are more rows than patients because the spreadsheet has one row per breast with a lesion: **LeftRight** says which breast the row describes. Only malignant cases from the second part of the collection (patient IDs starting with `D2`) have a molecular **subtype**.

In [ ]:
clinical["group"] = clinical["subtype"].fillna(clinical["classification"])
pd.crosstab(clinical["group"], clinical["ID1"].str[:2], margins=True)

# Join the clinical data to the image inventory

Since TCIA's public DICOM data are now served by NCI's Imaging Data Commons (IDC), we use the **idc** module of tcia_utils to list the image series. In CMMD each patient has one series, and its **PatientID** matches **ID1** in the spreadsheet.

**Note:** the TCIA collection page says that where a row has both an ID1 and an ID2, the image database stores only the ID2 value as PatientID. The revised spreadsheet we downloaded has only the ID1 column, and we check below that every ID1 matches a PatientID.

In [ ]:
series = idc.getSeries(collection="cmmd", format="df")
series = series[["PatientID", "SeriesInstanceUID", "ImageCount", "FileSizeMiB", "LicenseName"]]

print(len(series), "series for", series["PatientID"].nunique(), "patients")
print("Clinical rows whose ID1 has images:", clinical["ID1"].isin(series["PatientID"]).sum(), "of", len(clinical))

In [ ]:
patients = clinical.merge(series, left_on="ID1", right_on="PatientID", how="inner")
pd.crosstab(patients["group"], patients["ImageCount"].rename("images per patient"))

Most patients in the first part of the collection have two images, a craniocaudal (CC) and a mediolateral oblique (MLO) view of the breast with the lesion, and one clinical row. Most patients in the second part have four images, both views of both breasts, but still only one clinical row. To know which of their images the clinical data describes, we need to know which breast each image shows. That is stored in each image's DICOM header, so we will download a few patients and read it.

Let's pick the first patient for each molecular subtype who has one clinical row and a mass.

In [ ]:
rows_per_patient = clinical.groupby("ID1").size()
single_row = patients[patients["ID1"].map(rows_per_patient) == 1]

examples = (single_row[single_row["subtype"].notna() & (single_row["abnormality"] == "mass")]
            .sort_values("ID1")
            .groupby("subtype")
            .head(1)
            .reset_index(drop=True))

examples[["ID1", "LeftRight", "Age", "abnormality", "classification", "subtype", "ImageCount", "FileSizeMiB"]]

# Download the example images

**downloadSeries()** saves each series into its own folder named after its SeriesInstanceUID inside **tciaDownload**. These four series are about 70 MB in total.

In [ ]:
idc.downloadSeries(examples, input_type="df")

Now we read the header of every downloaded image. **ImageLaterality** gives the breast side (R or L), and **ViewCodeSequence** gives the view.

In [ ]:
images = []
for uid in examples["SeriesInstanceUID"]:
    for f in sorted(Path("tciaDownload", uid).glob("*.dcm")):
        ds = pydicom.dcmread(f, stop_before_pixels=True)
        images.append({
            "PatientID": ds.PatientID,
            "ImageLaterality": ds.ImageLaterality,
            "View": ds.ViewCodeSequence[0].CodeMeaning,
            "PatientAge": ds.PatientAge,
            "file": str(f),
        })
images = pd.DataFrame(images).sort_values(["PatientID", "ImageLaterality", "View"], ignore_index=True)

view_abbreviation = {"cranio-caudal": "CC", "medio-lateral oblique": "MLO"}
images["View"] = images["View"].map(view_abbreviation)
images[["PatientID", "ImageLaterality", "View", "PatientAge"]]

Joining on both the patient ID and the breast side attaches each clinical row to the two images of the breast it describes. The images of the other breast get no clinical values.

In [ ]:
joined = images.merge(
    clinical[["ID1", "LeftRight", "Age", "abnormality", "classification", "subtype"]],
    left_on=["PatientID", "ImageLaterality"],
    right_on=["ID1", "LeftRight"],
    how="left",
).drop(columns=["ID1", "LeftRight"])

joined.drop(columns="file")

# View the images with their clinical data

Each row below is one patient, with the right breast on the left and the left breast on the right, as on a radiology workstation. The panels of the breast described by the clinical spreadsheet are titled with its values.

In [ ]:
order = [("R", "CC"), ("R", "MLO"), ("L", "CC"), ("L", "MLO")]
fig, axes = plt.subplots(len(examples), 4, figsize=(12, 4 * len(examples)))

for row, patient_id in enumerate(examples["ID1"]):
    patient_images = joined[joined["PatientID"] == patient_id]
    for col, (side, view) in enumerate(order):
        ax = axes[row, col]
        ax.axis("off")
        match = patient_images[(patient_images["ImageLaterality"] == side) & (patient_images["View"] == view)]
        if match.empty:
            continue
        image = match.iloc[0]
        ax.imshow(pydicom.dcmread(image["file"]).pixel_array, cmap="gray")
        title = f"{patient_id} {side} {view}"
        if pd.notna(image["subtype"]):
            title += f"\n{image['subtype']}, {image['abnormality']}, age {int(image['Age'])}"
            ax.set_title(title, fontsize=10, fontweight="bold")
        else:
            ax.set_title(title + "\nno clinical row", fontsize=10, color="gray")

plt.tight_layout()
plt.show()

The joined table also works for the whole collection without downloading anything. For example, here is the patient age for each group in the clinical data.

In [ ]:
group_order = ["Benign", "Malignant", "Luminal A", "Luminal B", "HER2-enriched", "triple negative"]
ages = [patients.loc[patients["group"] == g, "Age"] for g in group_order]

fig, ax = plt.subplots(figsize=(9, 4))
ax.boxplot(ages)
ax.set_xticks(range(1, len(group_order) + 1), [f"{g}\n(n={len(a)})" for g, a in zip(group_order, ages)])
ax.set_ylabel("Age (years)")
ax.set_title("CMMD age by clinical group (one value per clinical row)")
plt.tight_layout()
plt.show()

# Notes

* "Malignant" in the chart means malignant without a recorded subtype.
* The TCIA collection page lists a few image pairs whose pixel data appear identical (D1-0202 and D2-0284, and D1-0808 and D1-1292). You may want to drop one of each pair before training a model.
* If you want to download the full collection, pass `series` to **downloadSeries()** instead of `examples`. It is about 22 GB.

# Acknowledgements
TCIA is funded by the [Cancer Imaging Program (CIP)](https://imaging.cancer.gov/), a part of the United States [National Cancer Institute (NCI)](https://www.cancer.gov/).  It is managed by the [Frederick National Laboratory for Cancer Research (FNLCR)](https://frederick.cancer.gov/) and hosted by the [University of Arkansas for Medical Sciences (UAMS)](https://www.uams.edu/)

This notebook was created by [Aaron Kanzer](https://github.com/aaronkanzer) (Scigantic). If you leverage this notebook or any TCIA datasets in your work, please be sure to comply with the [TCIA Data Usage Policy](https://wiki.cancerimagingarchive.net/x/c4hF). In particular, make sure to cite the DOI(s) for the specific TCIA datasets you used in addition to the following paper!

## TCIA Citation

Clark, K., Vendt, B., Smith, K., Freymann, J., Kirby, J., Koppel, P., Moore, S., Phillips, S., Maffitt, D., Pringle, M., Tarbox, L., & Prior, F. (2013). The Cancer Imaging Archive (TCIA): Maintaining and Operating a Public Information Repository. Journal of Digital Imaging, 26(6), 1045–1057. https://doi.org/10.1007/s10278-013-9622-7